In [29]:
import warnings
import os
from model_function import *
from model_utils import *
from utils import *
from torch.utils.data import DataLoader
import torch.nn.functional as Fin
import timeit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib 
from torchdiffeq import odeint as odeint
import matplotlib
matplotlib.use('Agg')
import argparse
import sys
import time
import torch
import torch.optim as optim
import random

In [30]:
levels = ["z","t","t2m","u10","v10"]
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
cwd = os.getcwd()
data_dir = '/mnt/nils'
spectral = 0
scale = 0

train_time_scale= slice('2006','2016')
val_time_scale = slice('2016','2016')
test_time_scale = slice('2017','2018')

In [31]:
paths_to_data = [str(data_dir) + '/era5_data/geopotential_500/*.nc',str(data_dir) + '/era5_data/temperature_850/*.nc',str(data_dir) + '/era5_data/2m_temperature/*.nc',str(data_dir) + '/era5_data/10m_u_component_of_wind/*.nc',str(data_dir) + '/era5_data/10m_v_component_of_wind/*.nc',str(data_dir) + '/era5_data/v_component_of_wind/*.nc',str(data_dir) + '/era5_data/u_component_of_wind/*.nc',str(data_dir) + '/era5_data/relative_humidity/*.nc',str(data_dir) + '/era5_data/specific_humidity/*.nc']
const_info_path = [str(data_dir) +  '/era5_data/constants/constants_5.625deg.nc']
levels = ["z","t","t2m","u10","v10"]
paths_to_data = paths_to_data[0:5]
levels = levels[0:5]
assert len(paths_to_data) == len(levels), "Paths to different type of data must be same as number of types of observations"
print("############################ Data is loading ###########################")
Final_train_data = 0
Final_val_data = 0
Final_test_data = 0
max_lev = []
min_lev = []
for idx,data in enumerate(paths_to_data):
    Train_data,Val_data,Test_data,time_steps,lat,lon,mean,std,time_stamp = get_train_test_data_without_scales_batched(data,train_time_scale,val_time_scale,test_time_scale,levels[idx],0)  
    max_lev.append(mean)
    min_lev.append(std)
    if idx==0: 
        Final_train_data = Train_data
        Final_val_data = Val_data
        Final_test_data = Test_data
    else:
        Final_train_data = torch.cat([Final_train_data,Train_data],dim=2)
        Final_val_data = torch.cat([Final_val_data,Val_data],dim=2)
        Final_test_data = torch.cat([Final_test_data,Test_data],dim=2)

print("Length of training data",len(Final_train_data))
print("Length of validation data",len(Final_val_data))
print("Length of testing data",len(Final_test_data))
const_channels_info,lat_map,lon_map = add_constant_info(const_info_path)

############################ Data is loading ###########################
Length of training data 1460
Length of validation data 1460
Length of testing data 1460


In [32]:
if spectral == 1: print("############## Running the Model in Spectral Domain ####################")
H,W = Train_data.shape[3],Train_data.shape[4]
clim = torch.mean(Final_test_data,dim=0)
Test_loader = DataLoader(Final_test_data[2:],batch_size=8,shuffle=False)
time_loader = DataLoader(time_steps[2:],batch_size=8,shuffle=False)
time_idx_steps = torch.tensor([i for i in range(365*4)]).view(-1,1)
time_idx = DataLoader(time_idx_steps[2:],batch_size=8,shuffle=False,pin_memory=False)

total_time_len = len(time_steps[2:])
total_time_steps = time_steps[2:].numpy().flatten().tolist()
num_years  = 2
Final_train_data = 0
Final_val_data = 0

In [33]:
def evaluate(model, vel_test, use_emission=True):
    R = {lev: [[] for _ in range(7)] for lev in levels}
    # A = {lev: [[] for _ in range(7)] for lev in levels}
    # C = {lev: [[] for _ in range(7)] for lev in levels}
    for entry, (time_steps, batch) in enumerate(zip(time_loader, Test_loader)):
        data = batch[0].to(device).view(num_years,1,len(paths_to_data)*(scale+1),H,W)
        past_sample = vel_test[entry].view(num_years,2*len(paths_to_data)*(scale+1),H,W).to(device)
        model.update_param([past_sample, const_channels_info.to(device),
                            lat_map.to(device), lon_map.to(device)])
        t = time_steps.float().to(device).flatten()
        with torch.no_grad():
            mean_pred, std_pred, mean_no_emission = model(t, data)
            if not use_emission:
                mean_pred = mean_no_emission

        for yr in range(2):
            for t_step in range(1, len(time_steps), 1):
                e_r = evaluation_rmsd_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                         lat,lon,max_lev,min_lev,H,W,levels)
                # e_a = evaluation_acc_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                        # lat,lon,max_lev,min_lev,H,W,levels,
                                        # clim[yr].cpu().detach().numpy())
                # if use_emission:
                    # e_c = evaluation_crps_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                            # lat,lon,max_lev,min_lev,H,W,levels,
                                            # std_pred[t_step,yr].cpu())
                for idx, lev in enumerate(levels):
                    R[lev][t_step-1].append(e_r[idx])
                    # A[lev][t_step-1].append(e_a[idx])
                    # if use_emission:
                        # C[lev][t_step-1].append(e_c[idx])
    # return {"RMSD": R, "ACC": A, "CRPS": C}
    return {"RMSD": R}

In [34]:
import re
from collections import defaultdict

LEADS = [(i+1)*6 for i in range(7)]

def plot_figure7(results, metric="RMSD", fname=None):
    # Seeds zu einem Modell zusammenfassen: "ClimODE (Baseline 42)" -> "ClimODE (Baseline)"
    groups = defaultdict(list)
    for label, res in results.items():
        name = re.sub(r"\s*\(\d+\)$", "", label)    # "ZeroModel (42)"        -> "ZeroModel"
        name = re.sub(r"\s+\d+\)$", ")", name)      # "ClimODE (Baseline 42)" -> "ClimODE (Baseline)"
        groups[name].append(res)

    fig, axes = plt.subplots(1, len(levels),)
    for ax, lev in zip(axes, levels):
        for (label, runs), col in zip(groups.items(), plt.cm.tab10.colors):
            vals = np.array([[np.mean(r[metric][lev][i]) for i in range(7)] for r in runs])  # (n_seeds, 7)
            m, s = vals.mean(0), vals.std(0)
            ax.errorbar(LEADS, m, yerr=s if len(runs) > 1 else None,
                        fmt="-o", ms=4, lw=1.8, capsize=6, color=col, label=label)
        ax.set_title(lev, fontsize=11)
        ax.set_xlabel("Lead time [h]")
        ax.set_xticks(LEADS)
        ax.grid(alpha=0.3, lw=0.6)
    axes[0].set_ylabel(f"latitude-weighted {metric}")
    axes[0].legend(frameon=False, fontsize=9)
    fig.tight_layout()
    # fig.savefig(fname or f"fig_{metric.lower()}.png", dpi=200, bbox_inches="tight")

In [35]:
LEADS = [(i + 1) * 6 for i in range(7)]

TITLES = {
    "z":   "z500",
    "t":   "t850",
    "t2m": "t2m",
    "u10": "u10",
    "v10": "v10",
}
LABELS = {  # Namen wie in der Caption
    "ClimODE reference":  "ClimODE (checkpoint)",
    "ClimODE (Baseline)": "ClimODE (retrained)",
}

def plot_figure7(results, levels, metric="RMSD", metric_label="RMSE", fname=None):
    groups = defaultdict(list)
    for label, res in results.items():
        name = re.sub(r"\s*\(\d+\)$", "", label)
        name = re.sub(r"\s+\d+\)$", ")", name)
        groups[name].append(res)

    fig, axes = plt.subplots(2, 3, sharex=True,
                             figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.55),
                             layout="constrained")
    axes = axes.ravel()

    for ax, lev in zip(axes, levels):
        for (label, runs), col in zip(groups.items(), plt.cm.tab10.colors):
            vals = np.array([[np.mean(r[metric][lev][i]) for i in range(7)] for r in runs])
            m = vals.mean(0)
            s = vals.std(0, ddof=1) if len(runs) > 1 else None
            ax.errorbar(LEADS, m, yerr=s, fmt="-o", ms=2.5, lw=1.0, capsize=4,
                        color=col, label=LABELS.get(label, label))
        ax.set_title(TITLES.get(lev, lev))
        ax.set_xticks(LEADS)

    # sharex blendet Ticklabels oben aus -> t2m (oben rechts) hat nichts darunter
    axes[2].tick_params(labelbottom=True)
    for ax in (axes[2], axes[3], axes[4]):
        ax.set_xlabel("Lead time [h]")
    for ax in (axes[0], axes[3]):
        ax.set_ylabel(f"latitude-weighted {metric_label}")

    # sechstes Feld = Legende
    axes[5].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    axes[5].legend(handles, labels, loc="center", frameon=False)

    return fig

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

LEADS = [(i + 1) * 6 for i in range(7)]
 
TITLES = {
    "z":   "z500",
    "t":   "t850",
    "t2m": "t2m",
    "u10": "u10",
    "v10": "v10",
}
 
STYLES = {
    "vet":          dict(color="tab:blue",   ls="-",  marker="o"),
    "lucas-kanade": dict(color="tab:green",  ls="--", marker="s"),
    "proesmans":    dict(color="tab:orange", ls=":",  marker="^"),
    "farneback":    dict(color="tab:pink",   ls="-.", marker="D"),
}

LABELS = {
    "Lucas-Kanade": "Lucas–Kanade",
    "Farneback":    "Farnebäck",
}
 
 
def _group_runs(results, with_seeds=False):
    groups = defaultdict(list)
    seeds = defaultdict(list)
    for label, res in results.items():
        name = re.sub(r"\s*\(\d+\)$", "", label)
        name = re.sub(r"\s+\d+\)$", ")", name)
        groups[name].append(res)
        m = re.search(r"\s(\d+)\)$", label)
        seeds[name].append(m.group(1) if m else None)
    return (groups, seeds) if with_seeds else groups
 
 
def _values(runs, metric, lev):
    return np.array([[np.mean(r[metric][lev][i]) for i in range(len(LEADS))]
                     for r in runs])
 
 
def plot_delta_rmse(results, levels, baseline="ClimODE (Baseline)",
                    exclude=("ClimODE reference",), paired=(),
                    metric="RMSD", sharey=False, textwidth_in=None, fname=None, bands=()):
    if textwidth_in is None:
        textwidth_in = TEXTWIDTH_IN  # globale Konstante aus deinem Skript
 
    groups, seeds = _group_runs(results, with_seeds=True)
    if baseline not in groups:
        raise KeyError(f"Baseline '{baseline}' nicht gefunden. "
                       f"Vorhanden: {list(groups)}")
    base_runs = groups.pop(baseline)
    base_seeds = seeds.pop(baseline)
    for name in exclude:
        groups.pop(name, None)
 
    pair_idx = {}
    for name in paired:
        if name not in groups:
            raise KeyError(f"Gepaarte Gruppe '{name}' nicht gefunden. "
                           f"Vorhanden: {list(groups)}")
        missing = [s for s in seeds[name] if s not in base_seeds]
        if missing or None in seeds[name]:
            raise ValueError(f"Seeds von '{name}' ({seeds[name]}) passen nicht "
                             f"zur Baseline ({base_seeds}).")
        pair_idx[name] = [base_seeds.index(s) for s in seeds[name]]
 
    fallback = plt.cm.tab10.colors
    styles = {}
    for k, name in enumerate(groups):
        styles[name] = STYLES.get(name) or dict(
            color=fallback[k % len(fallback)], ls="-", marker="o")
    unknown = [n for n in groups if n not in STYLES]
    if unknown:
        print(f"Kein Eintrag in STYLES für {unknown} -> Ersatzfarben. "
              f"Schlüssel in STYLES/LABELS an diese Namen anpassen.")
 
    fig, axes = plt.subplots(2, 3, sharex=True, sharey=sharey,
                             figsize=(textwidth_in, textwidth_in * 0.55),
                             layout="constrained")
    axes = axes.ravel()
 
    for ax, lev in zip(axes, levels):
        base = _values(base_runs, metric, lev)
        base_mean = base.mean(0)
        base_std = base.std(0, ddof=1) if len(base_runs) > 1 else np.zeros(len(LEADS))
        band = 100 * base_std / base_mean
 
        ax.fill_between(LEADS, -band, band, color="0.85", lw=0, zorder=0)
        ax.axhline(0, color="0.4", lw=0.8, zorder=1)
 
        for name, runs in groups.items():
            vals = _values(runs, metric, lev)
            if name in pair_idx:
                rel = 100 * (vals / base[pair_idx[name]] - 1) 
            else:
                rel = 100 * (vals / base_mean - 1)         
            m = rel.mean(0)
            s = rel.std(0, ddof=1) if len(runs) > 1 else None
            if name in bands and s is not None: 
                st = styles[name]
                ax.fill_between(LEADS, m - s, m + s, color=st["color"],
                                alpha=0.2, lw=0, zorder=1.5)
                ax.plot(LEADS, m, ms=2.5, lw=1.0, zorder=2,
                        label=LABELS.get(name, name), **st)
                continue
            ax.errorbar(LEADS, m, yerr=s, ms=2.5, lw=1.0,
                        capsize=3 if s is not None else 0,
                        label=LABELS.get(name, name), zorder=2, **styles[name])
 
        ax.set_title(TITLES.get(lev, lev))
        ax.set_xticks(LEADS)
 
    axes[2].tick_params(labelbottom=True)
    for ax in (axes[2], axes[3], axes[4]):
        ax.set_xlabel("Lead time [h]")
    for ax in (axes[0], axes[3]):
        ax.set_ylabel(r"$\Delta$RMSE to baseline [%]")
 
    axes[5].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    for i, lab in enumerate(labels):               
        name = next((n for n in bands if LABELS.get(n, n) == lab), None)
        if name is not None:
            c = styles[name]["color"]
            handles[i] = (Patch(facecolor=c, alpha=0.2, edgecolor="none"), handles[i])
    band_handle = (Patch(facecolor="0.85", edgecolor="none"),
                   Line2D([], [], color="0.4", lw=0.8))
    axes[5].legend([band_handle] + handles,
                   [r"Baseline ($\pm$1 std)"] + labels,
                   loc="center", frameon=False)
 
    if fname is not None:
        fig.savefig(fname)
    return fig

In [ ]:
TEXTWIDTH_IN = 6.496

plt.rcParams.update({
    "figure.figsize": (TEXTWIDTH_IN, TEXTWIDTH_IN * 0.62),
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,

    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",

    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.linewidth": 0.8,
})


In [ ]:
CONFIGS = {
    "fit-free ClimODE (42)": ("/Models/Zero_models/Zero_model_42/ClimODE_global_euler_0_model_298.pt", 
                        "None"),
    "fit-free ClimODE (123)": ("/Models/Zero_models/Zero_model_123/ClimODE_global_euler_0_model_297.pt", 
                        "None"),
    "fit-free ClimODE (456)": ("/Models/Zero_models/Zero_model_456/ClimODE_global_euler_0_model_299.pt", 
                        "None"),
    "fit-free ClimODE (789)": ("/Models/Zero_models/Zero_model_789/ClimODE_global_euler_0_model_299.pt", 
                            "None"),
    "ClimODE (Baseline 42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                            "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                                "initial_velocities/original_vel/original_vel_123/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_456/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_789/test_10year_2day_mm_vel.npy"),
    "farneback": ("/Models/Models_farneback/ClimODE_global_euler_0_model_299.pt",
                        "initial_velocities/optical_flow_vel_farneback/test_10year_2day_mm_farneback_vel.npy"),
    "proesmans":   ("/Models/Proesmans_model/Proesmans_model_42/ClimODE_global_euler_0_model_299.pt",
                           "initial_velocities/optical_flow_vel_proesmans/test_10year_2day_mm_proesmans_vel.npy"),
    "vet": ("/Models/Models_vet/ClimODE_global_euler_0_model_299.pt",
                   "initial_velocities/optical_flow_vel_vet/test_10year_2day_mm_VET_vel.npy"),
    "lucas-kanade": ("/Models/Models_lk/ClimODE_global_euler_0_model_299.pt",
                    "initial_velocities/optical_flow_vel_lk/test_10year_2day_mm_lk_vel.npy"),
}

results_all = {}
for label, (ckpt, velfile) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))          # v0 = 0
    else:
        vel_test = torch.from_numpy(np.load(velfile))
    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results_all[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
%matplotlib inline

plot_delta_rmse(results_all, levels, bands=("fit-free ClimODE"))
plt.savefig("of_vs_zero_training.pdf")
plt.show()

### Optical flow results (already in the thesis)

In [ ]:
CONFIGS = {
    "lucas-kanade": ("/Models/Models_lk/ClimODE_global_euler_0_model_299.pt",
                     "initial_velocities/optical_flow_vel_lk/test_10year_2day_mm_lk_vel.npy"),
    "ClimODE (Baseline 42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                            "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                                "initial_velocities/original_vel/original_vel_123/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_456/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_789/test_10year_2day_mm_vel.npy"),
    "farneback": ("/Models/Models_farneback/ClimODE_global_euler_0_model_299.pt",
                  "initial_velocities/optical_flow_vel_farneback/test_10year_2day_mm_farneback_vel.npy"),
    "proesmans":   ("/Models/Proesmans_model/Proesmans_model_42/ClimODE_global_euler_0_model_299.pt",
                    "initial_velocities/optical_flow_vel_proesmans/test_10year_2day_mm_proesmans_vel.npy"),
    "vet": ("/Models/Models_vet/ClimODE_global_euler_0_model_299.pt",
            "initial_velocities/optical_flow_vel_vet/test_10year_2day_mm_VET_vel.npy"),
}

results_optical_flow = {}
for label, (ckpt, velfile) in CONFIGS.items():
    vel_test = torch.from_numpy(np.load(velfile))
    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results_optical_flow[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
%matplotlib inline

plot_delta_rmse(results_optical_flow, levels)
plt.savefig("optical_flow_results.pdf")
plt.show()

### Basline model reproduction (already in thesis)

In [ ]:
CONFIGS = {
    "ClimODE reference": ("/checkpoints/ClimODE_global.pt", 
                               "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                           "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                               "initial_velocities/original_vel/original_vel_123/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                                   "initial_velocities/original_vel/original_vel_456/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                                   "initial_velocities/original_vel/original_vel_789/test_10year_2day_mm_vel.npy"),
}

results = {}
for label, (ckpt, velfile) in CONFIGS.items():
    vel_test = torch.from_numpy(np.load(velfile))
    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
%matplotlib inline

plot_figure7(results, levels, "RMSD")
# plot_figure7(results, "ACC")
plt.savefig("plots/reproduction.pdf")
plt.show()

In [ ]:
CONFIGS = {
    "ClimODE (Baseline 42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                            "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                                "initial_velocities/original_vel/original_vel_123/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_456/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_789/test_10year_2day_mm_vel.npy"),
    "With zero initial velocity (42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                        "None"),
    "With zero initial velocity (123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                      "None"),
    "With zero initial velocity (456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                      "None"),
    "With zero initial velocity (789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                      "None"),
}

results_zero_inference = {}
i = 0
for label, (ckpt, velfile) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))
        print(f"{label} loaded with zero velocity.")
    else:
        vel_test = torch.from_numpy(np.load(velfile))
        print(f"{label} loaded with velocity from file.")
    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results_zero_inference[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
%matplotlib inline

STYLES["With zero initial velocity"] = dict(color="#5E3C99", ls="-", marker="*")

plot_delta_rmse(results_zero_inference, levels, bands=("With zero initial velocity"))
plt.savefig("zero_flow_results.pdf")
plt.show()

### Last validation chapter (still under construction)

In [ ]:
CONFIGS = {
    "fit-free ClimODE (42)": ("/Models/Zero_models/Zero_model_42/ClimODE_global_euler_0_model_298.pt", 
                           "None"),
    "fit-free ClimODE (123)": ("/Models/Zero_models/Zero_model_123/ClimODE_global_euler_0_model_297.pt", 
                        "None"),
    "fit-free ClimODE (456)": ("/Models/Zero_models/Zero_model_456/ClimODE_global_euler_0_model_299.pt", 
                        "None"),
    "fit-free ClimODE (789)": ("/Models/Zero_models/Zero_model_789/ClimODE_global_euler_0_model_299.pt", 
                        "None"),
    "ClimODE (Baseline 42)": ("/Models/Base_models/Base_model_42/ClimODE_global_euler_0_model_299.pt", 
                               "initial_velocities/original_vel/original_vel_42/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 123)": ("/Models/Base_models/Base_model_123/ClimODE_global_euler_0_model_299.pt", 
                                "initial_velocities/original_vel/original_vel_123/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 456)": ("/Models/Base_models/Base_model_456/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_456/test_10year_2day_mm_vel.npy"),
    "ClimODE (Baseline 789)": ("/Models/Base_models/Base_model_789/ClimODE_global_euler_0_model_299.pt", 
                                    "initial_velocities/original_vel/original_vel_789/test_10year_2day_mm_vel.npy"),
}

results_zero_models = {}
for label, (ckpt, velfile) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))
    else:
        vel_test = torch.from_numpy(np.load(velfile))

    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results_zero_models[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
STYLES["fit-free ClimODE"] = dict(color="#E66101", ls="-", marker="o")

plot_delta_rmse(results_zero_models, levels, bands=('fit-free ClimODE'))
plt.savefig("zero_training_results.pdf")
plt.show()

In [ ]:
CONFIGS = {
    "ZeroModel (42)": ("/Models/Zero_models/Zero_model_42/ClimODE_global_euler_0_model_298.pt", 
                           "None"),
    "ZeroModel (123)": ("/Models/Zero_models/Zero_model_123/ClimODE_global_euler_0_model_297.pt", 
                        "None"),
    "ZeroModel (456)": ("/Models/Zero_models/Zero_model_456/ClimODE_global_euler_0_model_299.pt", 
                        "None"),
    "farneback ($-v$)": ("/Models/Models_farneback/ClimODE_global_euler_0_model_299.pt",
                            "initial_velocities/optical_flow_vel_farneback/test_10year_2day_mm_farneback_vel.npy"),
    "proesmans ($-v$)":   ("/Models/Proesmans_model/Proesmans_model_42/ClimODE_global_euler_0_model_299.pt",
                            "initial_velocities/optical_flow_vel_proesmans/test_10year_2day_mm_proesmans_vel.npy"),
    "vet ($-v$)": ("/Models/Models_vet/ClimODE_global_euler_0_model_298.pt",
                    "initial_velocities/optical_flow_vel_vet/test_10year_2day_mm_VET_vel.npy"),
}

results = {}
for label, (ckpt, velfile) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))
    else:
        vel_test = torch.from_numpy(np.load(velfile))

    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    results[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
CONFIGS = {
    "ClimODE": ("/checkpoints/ClimODE_global.pt", 
                           "None"),
    "ClimODE only adv": ("/checkpoints/ClimODE_global.pt", 
                           "None"),
}

results = {}
i = 0
for label, (ckpt, velfile) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))
    else:
        vel_test = torch.from_numpy(np.load(velfile))

    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    # model.eval()
    if i < 1:
        results[label] = evaluate(model, vel_test, use_emission=True)
    else:
        results[label] = evaluate(model, vel_test, use_emission=False)
    print("fertig:", label)
    i = i + 1

In [ ]:
%matplotlib inline

plot_figure7(results, levels)
# plot_figure7(results, "ACC")
plt.savefig("adv_only.pdf")
plt.show()